<div dir="rtl">

# פרויקט למידת מכונה - חלק ב'
## Airline Satisfaction - קבוצה 3

מחברת זו מציגה את שלבי הניתוח האמפירי של הנתונים באמצעות קוד וויזואליזציות.  
ההסברים התיאורטיים המלאים מופיעים בדו"ח הנלווה.

</div>

# עיבוד מקדים מתוקן בעקבות המשוב על חלק א׳

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
df = pd.read_csv('Xy_train.csv')
df_clean = df.copy()
df.head()

,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Plane colors,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,...,Seat comfort,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,Female,Loyal Customer,35.0,Personal Travel,Eco,731.0,2,4.0,3.0,1.0,...,3.0,4.0,NaN,5.0,1.0,5.0,2.0,18.0,1.0,neutral or dissatisfied
1,Female,Loyal Customer,35.0,Business travel,Unknown,354.0,2,3.0,3.0,3.0,...,3.0,2.0,3.0,3.0,1.0,3.0,3.0,0.0,0.0,neutral or dissatisfied
2,Male,disloyal Customer,43.0,Business travel,Eco,719.0,3,2.0,2.0,2.0,...,2.0,1.0,3.0,1.0,2.0,4.0,4.0,0.0,0.0,neutral or dissatisfied
3,Male,disloyal Customer,21.0,Business travel,Eco,772.0,1,2.0,2.0,2.0,...,2.0,3.0,3.0,3.0,3.0,5.0,2.0,11.0,7.0,neutral or dissatisfied
4,Male,Loyal Customer,39.0,Personal Travel,Eco,618.0,1,4.0,2.0,4.0,...,2.0,4.0,NaN,3.0,3.0,5.0,2.0,0.0,0.0,neutral or dissatisfied


In [ ]:
df.shape

(9000, 22)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9000 entries, 0 to 8999
Data columns (total 22 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Gender                             9000 non-null   object 
 1   Customer Type                      8999 non-null   object 
 2   Age                                8999 non-null   float64
 3   Type of Travel                     8999 non-null   object 
 4   Class                              8999 non-null   object 
 5   Flight Distance                    8998 non-null   float64
 6   Plane colors                       9000 non-null   int64  
 7   Inflight wifi service              8998 non-null   float64
 8   Departure/Arrival time convenient  8998 non-null   float64
 9   Ease of Online booking             8998 non-null   float64
 10  Gate location                      8998 non-null   float64
 11  Food and drink                     8998 non-null   float

<div dir="rtl">

#  הכנת נתונים (Data Preprocessing)

</div>

<div dir="rtl">
<mark>
### טיפול מתוקן בערכים חריגים וחסרים בעקבות המשוב
<mark>
בעקבות המשוב על חלק א׳, עודכנה הגישה לטיפול בערכים בעייתיים. במקום למחוק רשומות שלמות בשל ערך חריג במאפיין בודד, הערך הבעייתי בלבד הומר ל־NaN והושלם בהמשך בשיטה מתאימה. חריגה אחת היא רשומה שבה הופיע טקסט חופשי ולא תקין במשתנה `Class` במקרה זה הוחלט להסיר את הרשומה, משום שמדובר בשגיאת הזנה ברורה שעלולה להעיד על חוסר מהימנות של הרשומה כולה.

</div>

In [ ]:
print("Original rows:", df.shape[0])
print("df_clean rows:", df_clean.shape[0])

Original rows: 9000
df_clean rows: 9000


### זיהוי והסרת רשומה עם ערך לא תקין במשתנה Class

In [ ]:
# זיהוי ערכים לא תקינים ב-Class
valid_classes = ["Eco", "Eco Plus", "Business", "Unknown"]

invalid_class_rows = df_clean[
    ~df_clean["Class"].isin(valid_classes)
]

print("Invalid Class rows:")
display(invalid_class_rows[["Class", "Type of Travel", "satisfaction"]])

print("Number of invalid Class rows:", invalid_class_rows.shape[0])

# הסרת הרשומה/ות עם ערך טקסטואלי לא תקין ב-Class
df_clean = df_clean.drop(index=invalid_class_rows.index)

print("Rows after removing invalid Class rows:", df_clean.shape[0])

Invalid Class rows:


,Class,Type of Travel,satisfaction
5568,NaN,NaN,satisfied
8732,IT IS SO BORING WORKING IN AN AIRPORT'S DESK O...,Business travel,satisfied


Number of invalid Class rows: 2
Rows after removing invalid Class rows: 8998


ממצא זה חושף כי המאגר אינו תוצר של מערכת ממוחשבת חסינה, אלא כלל שלב של הזנה ידנית שאפשרה לעובדים "לפרוק תסכול" לתוך הדאטה.
רשומה כזו מעידה על חוסר מהימנות של יתר הערכים באותה שורה ומומלץ להשמיט את הרשומה כולה מתוך הדאטה ומהניתוח.


### בחינת ערכי Unknown במשתנה Class לפי Type of Travel

<div dir="rtl">

במשתנה <b>Class</b> קיימים ערכי <b>Unknown</b>, שאינם מייצגים מחלקת טיסה אמיתית ולכן יטופלו כערכים חסרים. כדי לבחור דרך השלמה סבירה, בדקנו כיצד מתפלגות הרשומות שבהן <b>Class = Unknown</b> לפי המשתנה <b>Type of Travel</b>.  
מהבדיקה עולה שרוב הרשומות עם <b>Unknown</b> שייכות לקטגוריית <b>Business travel</b>, ולכן יש הצדקה להשתמש ב־<b>Type of Travel</b> כמשתנה עזר להשלמת הערכים החסרים ב־<b>Class</b>.

</div>

In [ ]:
unknown_class = df[df["Class"] == "Unknown"]

unknown_class["Type of Travel"].value_counts()

,count
Type of Travel,
Business travel,921
Personal Travel,390


In [ ]:
# החלפת כל ערכי Unknown במשתנה Class ל-Business
df_clean.loc[df_clean["Class"] == "Unknown", "Class"] = "Business"

# בדיקה שלא נשארו ערכי Unknown
print("Number of Unknown values after replacement:")
print((df_clean["Class"] == "Unknown").sum())

# בדיקת התפלגות Class לאחר ההחלפה
df_clean["Class"].value_counts()

Number of Unknown values after replacement:
0


,count
Class,
Business,4498
Eco,3942
Eco Plus,558


<div dir="rtl">

<mark>
בחלק א׳ שקלנו להסיר את המשתנה <b>Gate location</b>, מאחר שהקורלציה הפשוטה שלו עם משתנה המטרה <b>satisfaction</b> הייתה נמוכה. עם זאת, בעקבות המשוב שקיבלנו, בחנו את המשתנה מחדש לפי התפלגות שביעות הרצון בכל רמת דירוג. בבדיקה זו ניתן לראות כי קיימים הבדלים באחוזי שביעות הרצון בין קבוצות הדירוג השונות. לכן, הוחלט שלא להסיר את המשתנה מראש, אלא להשאירו בשלב זה ולבחון את תרומתו במסגרת המודלים.
</mark>

</div>

In [ ]:

gate_percent = pd.crosstab(
    df["Gate location"],
    df["satisfaction"],
    normalize="index"
) * 100

gate_percent.round(2)

satisfaction,neutral or dissatisfied,satisfied
Gate location,,
1.0,52.95,47.05
2.0,52.50,47.50
3.0,63.76,36.24
4.0,59.80,40.20
5.0,47.88,52.12
999.0,0.00,100.00


## <mark> טיפול בערך חריג במשתנה Gate location
<div dir="rtl">

במשתנה <b>Gate location</b> זוהה ערך חריג 999, שאינו נמצא בטווח הדירוג התקין של 1–5. בהתאם לגישה שנבחרה בעקבות המשוב, לא נמחקה הרשומה כולה, אלא הערך החריג בלבד הומר ל־<code>NaN</code> והושלם לאחר מכן באמצעות החציון של המשתנה. מכיוון שמדובר במשתנה דירוג אורדינלי, שימוש בחציון מתאים יותר מממוצע ושומר על ערך בתחום הסקאלה.

</div>

In [ ]:
# זיהוי ערכים חריגים ב-Gate location
invalid_gate_mask = ~df_clean["Gate location"].between(1, 5) & df_clean["Gate location"].notna()

print("Invalid Gate location values before treatment:", invalid_gate_mask.sum())

display(df_clean.loc[
    invalid_gate_mask,
    ["Gate location", "Class", "Type of Travel", "satisfaction"]
])

# הפיכת הערך החריג בלבד ל-NaN
df_clean.loc[invalid_gate_mask, "Gate location"] = np.nan

# השלמה באמצעות חציון
gate_median = df_clean["Gate location"].median()
df_clean["Gate location"] = df_clean["Gate location"].fillna(gate_median)

print("Gate location median used for imputation:", gate_median)
print("Invalid Gate location values after treatment:", (~df_clean["Gate location"].between(1, 5)).sum())
print("Missing Gate location values after treatment:", df_clean["Gate location"].isna().sum())

Invalid Gate location values before treatment: 1


,Gate location,Class,Type of Travel,satisfaction
7583,999.0,Business,Business travel,satisfied


Gate location median used for imputation: 3.0
Invalid Gate location values after treatment: 0
Missing Gate location values after treatment: 0


<div dir="rtl">

## השלמת ערכים חסרים– Leg room service

המשתנה Leg room service מכיל כ־30% ערכים חסרים. למרות שיעור החסר הגבוה יחסית, בחרנו שלא להשמיט אותו, משום שמדובר במאפיין שירות בעל משמעות תחומית ישירה לחוויית הנוסע. ניתוח החוסרים הראה כי הם אינם תלויים באופן מהותי במשתנה המטרה או במאפיינים אחרים, ולכן אין אינדיקציה חזקה לכך שמדובר בהטיה שיטתית.
במקום למחוק את המשתנה, בוצעה השלמה מושכלת של הערכים החסרים באופן היררכי: תחילה לפי Class ו־Type of Travel, לאחר מכן לפי Class בלבד, ולבסוף באמצעות חציון כללי במקרה הצורך. גישה זו מאפשרת לשמר שונות רלוונטית בין קבוצות נוסעים ולמנוע אובדן מידע חשוב.

</div>

<div dir="rtl">
<mark>
בעקבות המשוב על חלק א׳, הוספנו בדיקה מפורשת של הקשר בין החוסרים במשתנה Leg room service לבין משתנה המטרה. לשם כך יצרנו משתנה עזר בינארי שמציין האם הערך חסר, ובחנו את התפלגות satisfaction עבור רשומות שבהן הערך קיים לעומת רשומות שבהן הערך חסר. בדיקה זו מאפשרת לראות האם החוסרים מתרכזים באופן חריג באחת ממחלקות שביעות הרצון, לפני ביצוע ההשלמה.

</div>

In [ ]:
# בדיקה פשוטה: מתוך הרשומות שבהן חסר Leg room service,
# כמה satisfied וכמה neutral or dissatisfied

missing_leg = df_clean[df_clean['Leg room service'].isna()]

missing_leg_summary = (
    missing_leg['satisfaction']
    .value_counts()
    .reset_index()
)

missing_leg_summary.columns = ['Satisfaction', 'Count']

missing_leg_summary['Percent'] = (
    missing_leg_summary['Count'] / missing_leg_summary['Count'].sum() * 100
).round(2)

display(missing_leg_summary)

,Satisfaction,Count,Percent
0,neutral or dissatisfied,1525,56.48
1,satisfied,1175,43.52


<div dir="rtl">
<mark>
מהטבלה ניתן לראות שהתפלגות משתנה המטרה בקרב רשומות שבהן הערך ב-Leg room service חסר דומה יחסית להתפלגות בקרב רשומות שבהן הערך קיים. לכן, לא נראית אינדיקציה חזקה לכך שהחוסר תלוי באופן מהותי במשתנה המטרה. בהתאם לכך, המשכנו להשלמה היררכית לפי Class ו-Type of Travel, לאחר מכן לפי Class בלבד, ולבסוף לפי חציון כללי. זה הדרדור (?)

</div>

In [ ]:

# שלב 1: חציון לפי Class ו-Type of Travel
group_median_1 = df_clean.groupby(['Class', 'Type of Travel'])['Leg room service'].transform('median')

# שלב 2: חציון לפי Class
group_median_2 = df_clean.groupby('Class')['Leg room service'].transform('median')

# שלב 3: חציון כללי
global_median = df_clean['Leg room service'].median()

# ביצוע ההשלמה על גבי df_clean
df_clean['Leg room service'] = df_clean['Leg room service'].fillna(group_median_1)
df_clean['Leg room service'] = df_clean['Leg room service'].fillna(group_median_2)
df_clean['Leg room service'] = df_clean['Leg room service'].fillna(global_median)

print(f"Missing values in Leg room service after hierarchical imputation: {df_clean['Leg room service'].isna().sum()}")

Missing values in Leg room service after hierarchical imputation: 0


<div dir="rtl">

במשתנה `Leg room service`  לפני ההשלמה נבדקה התפלגות משתנה המטרה בקרב הרשומות שבהן חסר הערך, כדי לוודא שהחוסרים אינם מרוכזים באופן חריג באחת ממחלקות שביעות הרצון. לאחר מכן בוצעה השלמה היררכית: תחילה לפי `Class` ו־`Type of Travel`, לאחר מכן לפי `Class` בלבד, ולבסוף לפי החציון הכללי.

</div>

<div dir="rtl">

## 3.1.3.1. זיהוי והשלמת שורות בעייתיות – Age
נחפש ערכים שאינם הגיוניים ביולוגית (למשל גיל מתחת ל-0 או מעל 110) שעלולים להעיד על שגיאות בהקלדת הנתונים.
</div>

<mark>בדוח כתבנו שאין חריגים אבל בקולאב יש חריגים, והסרנו אותם



In [ ]:
# זיהוי ערכי Age חריגים
invalid_age_mask = (df_clean["Age"] < 0) | (df_clean["Age"] > 110)

print("Invalid Age values before treatment:", invalid_age_mask.sum())

display(df_clean.loc[invalid_age_mask, ["Age", "Class", "Type of Travel", "satisfaction"]])

# הפיכת הערך הבעייתי בלבד ל-NaN
df_clean.loc[invalid_age_mask, "Age"] = np.nan

# השלמה לפי חציון
age_median = df_clean["Age"].median()
df_clean["Age"] = df_clean["Age"].fillna(age_median)

print("Age median used for imputation:", age_median)
print("Invalid Age values after treatment:", ((df_clean["Age"] < 0) | (df_clean["Age"] > 110)).sum())
print("Missing Age values after treatment:", df_clean["Age"].isna().sum())

Invalid Age values before treatment: 2


,Age,Class,Type of Travel,satisfaction
7034,157.0,Eco,Business travel,satisfied
7035,156.0,Business,Business travel,satisfied


Age median used for imputation: 41.0
Invalid Age values after treatment: 0
Missing Age values after treatment: 0


<mark> בהתאם למשוב, לא הוסרו רשומות שלמות בשל ערך בעייתי בודד. ערכים חריגים או לא תקינים הומרו ל־NaN והושלמו בהמשך בשיטה מתאימה, למשל השלמת Age לפי חציון/ממוצע/חציון קבוצתי. כך ניתן לשמר את המידע התקין בשאר מאפייני הרשומה ולהימנע מאובדן מידע מיותר.


 <mark>  זיהוי והשלמת ערכים בשורות בעייתיות – Flight Distance מרחק טיסה אינו יכול להיות שלילי. נזהה ונשלים תצפיות אלו.


In [ ]:
# זיהוי מרחקי טיסה שליליים
negative_distance_mask = df_clean["Flight Distance"] < 0

print("Negative Flight Distance values before treatment:", negative_distance_mask.sum())

display(df_clean.loc[negative_distance_mask, ["Flight Distance", "Class", "Type of Travel", "satisfaction"]])

# הפיכת הערך השלילי בלבד ל-NaN
df_clean.loc[negative_distance_mask, "Flight Distance"] = np.nan

# השלמה היררכית:
# שלב 1: חציון לפי Class ו-Type of Travel
distance_median_group_1 = df_clean.groupby(["Class", "Type of Travel"])["Flight Distance"].transform("median")

# שלב 2: חציון לפי Class בלבד
distance_median_group_2 = df_clean.groupby("Class")["Flight Distance"].transform("median")

# שלב 3: חציון כללי
distance_global_median = df_clean["Flight Distance"].median()

df_clean["Flight Distance"] = df_clean["Flight Distance"].fillna(distance_median_group_1)
df_clean["Flight Distance"] = df_clean["Flight Distance"].fillna(distance_median_group_2)
df_clean["Flight Distance"] = df_clean["Flight Distance"].fillna(distance_global_median)

print("Global Flight Distance median used if needed:", distance_global_median)
print("Negative Flight Distance values after treatment:", (df_clean["Flight Distance"] < 0).sum())
print("Missing Flight Distance values after treatment:", df_clean["Flight Distance"].isna().sum())

Negative Flight Distance values before treatment: 1


,Flight Distance,Class,Type of Travel,satisfaction
4859,-204.0,Business,Business travel,satisfied


Global Flight Distance median used if needed: 853.5
Negative Flight Distance values after treatment: 0
Missing Flight Distance values after treatment: 0


<div dir="rtl">

## 3.1.1. השמטת מאפיין רועש – Plane colors
בדיקה מוקדמת העלתה כי המשתנה `Plane colors` אינו מציג קשר סטטיסטי מובהק למשתנה המטרה. נבצע בדיקת מתאם והתפלגות כדי לאשש זאת.
</div>

In [ ]:
# בחינת קשר למשתנה המטרה
ct = pd.crosstab(df['Plane colors'], df['satisfaction'], normalize='index')
display(ct)

# הסרת המשתנה כיוון שהוא אינו תורם ללמידה ומהווה 'רעש'
df_clean = df.drop(columns=['Plane colors'])
print("Feature 'Plane colors' dropped.")

satisfaction,neutral or dissatisfied,satisfied
Plane colors,,
1,0.571285,0.428715
2,0.560626,0.439374
3,0.558644,0.441356


Feature 'Plane colors' dropped.


<div dir="rtl">

## 3.1.4.1. השלמת ערכים חסרים במאפיינים רציפים
עבור משתנים כמותיים שנותרו, נבצע השלמה באמצעות חציון (Median), כיוון שהוא עמיד יותר לערכים חריגים מאשר הממוצע.
</div>

In [ ]:
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df_clean[col].isna().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print("Numeric columns imputed with median.")

Numeric columns imputed with median.


<div dir="rtl">

## 3.1.4.2. השלמת ערכים חסרים במאפיינים קטגוריאליים
עבור משתנים קטגוריאליים, נשתמש בשכיח (Mode) להשלמת הערכים החסרים.
</div>

In [ ]:
categorical_cols = df_clean.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df_clean[col].isna().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print("Categorical columns imputed with mode.")

Categorical columns imputed with mode.


<div dir="rtl">

### בדיקת סיכום לאחר שלבי הניקוי והעיבוד המקדים

בדיקה זו מסכמת את שלבי הטיפול שבוצעו בנתונים לפני אימון המודלים: הסרת הרשומה שבה הופיע ערך טקסטואלי חריג במשתנה <b>Class</b>, החלפת ערכי <b>Unknown</b> ב־<b>Business</b>, טיפול בערך החריג 999 במשתנה <b>Gate location</b>, טיפול בערכים חריגים במשתנים <b>Age</b> ו־<b>Flight Distance</b> ללא מחיקת רשומות שלמות, הסרת המשתנה <b>Plane colors</b>, והשלמת ערכים חסרים שנותרו במשתנים מספריים וקטגוריאליים.

מטרת הבדיקה היא לוודא שלא נותרו ערכים חסרים או לא תקינים לפני שלב המידול, וששלבי העיבוד בוצעו באופן עקבי.

</div>

In [ ]:
# בדיקת סיכום לאחר כל שלבי הניקוי והעיבוד המקדים

print("Summary check after preprocessing")
print("=" * 50)

print("""
This summary includes:
1. Removing the row with the invalid text value in Class:
   "IT IS SO BORING WORKING IN AN AIRPORT'S DESK OH MY GODDDDD"

2. Replacing Class = Unknown with Business.

3. Treating the invalid value 999 in Gate location.

4. Treating invalid values in Age and Flight Distance without removing full rows.

5. Removing the Plane colors variable.

6. General imputation of remaining missing values:
   numeric variables with median, categorical variables with mode.
""")

print("=" * 50)

# מספר שורות סופי
print("Final number of rows:", df_clean.shape[0])
print("Rows removed total:", df.shape[0] - df_clean.shape[0])

# בדיקת ערכים חסרים שנותרו
print("\nMissing values after treatments:")
remaining_missing = df_clean.isna().sum()
display(remaining_missing[remaining_missing > 0])

# בדיקת ערכי Class לאחר הטיפול
print("\nClass values after treatment:")
display(df_clean["Class"].value_counts(dropna=False))

# בדיקה שלא נשאר Unknown ב-Class
print("\nUnknown values in Class after treatment:")
print((df_clean["Class"] == "Unknown").sum())

# בדיקה שהטקסט החריג ב-Class לא נשאר
invalid_text = "IT IS SO BORING WORKING IN AN AIRPORT'S DESK OH MY GODDDDD"

print("\nInvalid text value in Class after treatment:")
print((df_clean["Class"] == invalid_text).sum())

# בדיקת Age
print("\nInvalid Age values after treatment:")
print(((df_clean["Age"] < 0) | (df_clean["Age"] > 110)).sum())

# בדיקת Flight Distance
print("\nNegative Flight Distance values after treatment:")
print((df_clean["Flight Distance"] < 0).sum())

# בדיקת Gate location
print("\nInvalid Gate location values after treatment:")
print((~df_clean["Gate location"].between(1, 5)).sum())

# בדיקת Leg room service
print("\nInvalid Leg room service values after treatment:")
print((~df_clean["Leg room service"].between(1, 5)).sum())

# בדיקת Plane colors
print("\nPlane colors column exists after treatment:")
print("Plane colors" in df_clean.columns)

Summary check after preprocessing

This summary includes:
1. Removing the row with the invalid text value in Class:
   "IT IS SO BORING WORKING IN AN AIRPORT'S DESK OH MY GODDDDD"

2. Replacing Class = Unknown with Business.

3. Treating the invalid value 999 in Gate location.

4. Treating invalid values in Age and Flight Distance without removing full rows.

5. Removing the Plane colors variable.

6. General imputation of remaining missing values:
   numeric variables with median, categorical variables with mode.

Final number of rows: 9000
Rows removed total: 0

Missing values after treatments:


,0



Class values after treatment:


,count
Class,
Eco,3943
Business,3187
Unknown,1311
Eco Plus,558
IT IS SO BORING WORKING IN AN AIRPORT'S DESK OH MY GODDDDD,1



Unknown values in Class after treatment:
1311

Invalid text value in Class after treatment:
1

Invalid Age values after treatment:
2

Negative Flight Distance values after treatment:
1

Invalid Gate location values after treatment:
1

Invalid Leg room service values after treatment:
28

Plane colors column exists after treatment:
False


<div dir="rtl">

### 3.2 טיפול פרטני במאפיינים

בשלב זה ביצענו דיסקרטיזציה של משתנים רציפים וגזירת מאפיינים חדשים מתוך הדאטה הנקי, במטרה לייצג את הנתונים בצורה נוחה וברורה יותר לניתוח.

</div>

In [ ]:
# 1. דיסקרטיזציה של גיל (Age) לקטגוריות בעלות משמעות עסקית
df_clean['Age_Category'] = pd.cut(
    df_clean['Age'],
    bins=[0, 12, 18, 65, np.inf],
    labels=['Child', 'Teen', 'Adult', 'Senior'],
    include_lowest=True
)

# 2. דיסקרטיזציה של מרחק טיסה (Flight Distance) לסוגי טיסות
df_clean['Flight_Type'] = pd.cut(
    df_clean['Flight Distance'],
    bins=[0, 1000, 3000, df_clean['Flight Distance'].max()],
    labels=['Short-Haul', 'Medium-Haul', 'Long-Haul'],
    include_lowest=True
)

# 3. גזירת מאפיין חדש: ציון שירות כולל (Total Service Score)
service_columns = [
    'Inflight wifi service',
    'Departure/Arrival time convenient',
    'Ease of Online booking',
    'Gate location',
    'Food and drink',
    'Seat comfort',
    'On-board service',
    'Leg room service',
    'Baggage handling',
    'Checkin service',
    'Inflight service',
    'Cleanliness'
]

df_clean['Total_Service_Score'] = df_clean[service_columns].mean(axis=1)

# בדיקה שהעמודות התווספו בהצלחה
print("New columns added:")
display(df_clean[['Age_Category', 'Flight_Type', 'Total_Service_Score']].head())

# בדיקה שאין ערכים חסרים בעמודות החדשות
print("\nMissing values in new columns:")
print(df_clean[['Age_Category', 'Flight_Type', 'Total_Service_Score']].isna().sum())

New columns added:


,Age_Category,Flight_Type,Total_Service_Score
0,Adult,Short-Haul,3.166667
1,Adult,Short-Haul,2.750000
2,Adult,Short-Haul,2.666667
3,Adult,Short-Haul,2.583333
4,Adult,Short-Haul,3.000000



Missing values in new columns:
Age_Category           0
Flight_Type            1
Total_Service_Score    0
dtype: int64
